# Polymorphism in Java
**Polymorphism** means "many forms" (*poly* = many, *morph* = forms). It allows an entity (like a method or object) to behave differently based on the context.

### Types of Polymorphism in Java:
1. **Compile-Time Polymorphism (Static Binding / Method Overloading)**
2. **Runtime Polymorphism (Dynamic Binding / Method Overriding)**


---
## 1. Compile-Time Polymorphism (Method Overloading)
- Multiple methods in the **same class** with the **same name** but **different parameter lists**.
- Parameters can differ by: **number of arguments**, **data types**, or **sequence of types**.
- Resolved at compile time by the compiler.
> Note: Changing only the return type does **not** overload a method in Java.


In [ ]:
class Calculator {
    // 1. Overloading by number of parameters
    int add(int a, int b) {
        return a + b;
    }

    int add(int a, int b, int c) {
        return a + b + c;
    }

    // 2. Overloading by data type of parameters
    double add(double a, double b) {
        return a + b;
    }
}

Calculator calc = new Calculator();
System.out.println("add(10, 20)       : " + calc.add(10, 20));
System.out.println("add(10, 20, 30)   : " + calc.add(10, 20, 30));
System.out.println("add(12.5, 7.5)    : " + calc.add(12.5, 7.5));


---
## 2. Runtime Polymorphism (Method Overriding / Dynamic Method Dispatch)
- A **subclass** provides a specific implementation of a method already defined in its **superclass**.
- **Rules**:
  - Method name, return type, and parameters must match exactly.
  - Requires an inheritance relationship (IS-A).
  - Uses **Dynamic Method Dispatch**: A call to an overridden method is resolved at runtime based on the actual object created (`Parent ref = new Child()`), not the reference type.


In [ ]:
// Superclass
class Animal {
    void makeSound() {
        System.out.println("Animal makes a sound.");
    }
}

// Subclasses overriding makeSound()
class Dog extends Animal {
    @Override
    void makeSound() {
        System.out.println("Dog barks: Woof Woof!");
    }
}

class Cat extends Animal {
    @Override
    void makeSound() {
        System.out.println("Cat meows: Meow Meow!");
    }
}

// Dynamic Method Dispatch (Parent reference -> Child object)
Animal myAnimal; // Parent reference

myAnimal = new Dog();
myAnimal.makeSound(); // Calls Dog's overridden method at runtime

myAnimal = new Cat();
myAnimal.makeSound(); // Calls Cat's overridden method at runtime


---
## Summary: Overloading vs Overriding

| Feature | Compile-Time (Overloading) | Runtime (Overriding) |
| :--- | :--- | :--- |
| **Definition** | Same name, different parameters | Same name, exact same parameters |
| **Location** | Within the **same class** | Across **parent and child classes** |
| **Binding** | Static binding (at compile-time) | Dynamic binding (at runtime) |
| **Inheritance** | Not required | Required (IS-A relationship) |
| **Return Type** | Can be same or different | Must be same (or covariant) |


---
## 3. Upcasting, Downcasting, and the `instanceof` Operator
- **Upcasting (Widening)**: Casting a subclass reference to a superclass reference (`Parent p = new Child();`). It is safe and done **implicitly** by Java.
- **Downcasting (Narrowing)**: Casting a superclass reference back to its concrete subclass (`Child c = (Child) p;`). Requires explicit cast and can throw `ClassCastException` if types do not match.
- **`instanceof` Operator**: Tests whether an object reference is an instance of a specific class or interface before downcasting.
- **Pattern Matching for `instanceof` (Modern Java 16+)**: Automatically binds the casted variable in one clean step (`if (obj instanceof Child c)`).


In [ ]:
class Device {
    void powerOn() { System.out.println("Device powered on."); }
}

class Smartphone extends Device {
    @Override
    void powerOn() { System.out.println("Smartphone booting OS..."); }
    void makeCall() { System.out.println("Making phone call..."); }
}

class Laptop extends Device {
    void compileCode() { System.out.println("Compiling Java code..."); }
}

// 1. Upcasting (Implicit)
Device myDevice = new Smartphone(); 
myDevice.powerOn(); // Calls overridden method in Smartphone

// 2. Traditional instanceof check before Downcasting
if (myDevice instanceof Smartphone) {
    Smartphone phone = (Smartphone) myDevice; // explicit downcast
    phone.makeCall();
}

// 3. Modern Pattern Matching for instanceof (Java 16+)
// Eliminates the boilerplate downcast line!
Device anotherDevice = new Laptop();
if (anotherDevice instanceof Laptop lap) {
    lap.compileCode(); // 'lap' is immediately available in scope
}


---
## 4. `final` Keyword and Sealed Classes (Java 17+)
- **`final` Keyword**:
  - `final variable`: Constant (cannot be reassigned).
  - `final method`: Cannot be overridden by subclasses.
  - `final class`: Cannot be inherited/extended (`String`, `Integer` are final classes).

- **`sealed` Classes (Modern Java 17+)**:
  - Allows an author to restrict which specific classes can extend or implement it using the `permits` keyword.
  - Subclasses must be declared as `final`, `sealed`, or `non-sealed`.


In [ ]:
// Sealed Class Hierarchy (Permits only specified subclasses)
sealed abstract class PaymentMethod permits CreditCard, UPI, NetBanking {
    abstract void processPayment(double amount);
}

final class CreditCard extends PaymentMethod {
    void processPayment(double amount) {
        System.out.println("Processing credit card payment: Rs. " + amount);
    }
}

final class UPI extends PaymentMethod {
    void processPayment(double amount) {
        System.out.println("Processing UPI payment: Rs. " + amount);
    }
}

final class NetBanking extends PaymentMethod {
    void processPayment(double amount) {
        System.out.println("Processing NetBanking payment: Rs. " + amount);
    }
}

PaymentMethod payment = new UPI();
payment.processPayment(1500.0);


---
## 5. Reflection API (`java.lang.reflect`)
**Reflection** allows inspecting and invoking classes, methods, and fields at runtime—even private members.

- `obj.getClass()` or `Class.forName()` gets class metadata.
- `getDeclaredFields()` and `getDeclaredMethods()` inspect members.
- `setAccessible(true)` allows accessing private members dynamically.


In [ ]:
import java.lang.reflect.Field;
import java.lang.reflect.Method;

class SecretAgent {
    private String secretCode = "AGENT-007";
    
    private void revealMission(String mission) {
        System.out.println("Mission: " + mission);
    }
}

// Using Reflection to inspect and access private members
SecretAgent agent = new SecretAgent();
Class<?> clazz = agent.getClass();

System.out.println("Class Name: " + clazz.getName());

// 1. Inspect and read private field
Field field = clazz.getDeclaredField("secretCode");
field.setAccessible(true); // bypass private access control
System.out.println("Extracted Private Field: " + field.get(agent));

// 2. Invoke private method
Method method = clazz.getDeclaredMethod("revealMission", String.class);
method.setAccessible(true);
method.invoke(agent, "Operation Top Secret");


---
## 6. Constant Pools in Java (Topic 45)
In Java, **Constant Pools** are dedicated optimization tables used by the compiler and JVM to store literal values and symbolic references to save memory and improve lookup speed:

1. **Class File Constant Pool**: Stored statically inside each compiled `.class` file. Contains symbolic names of classes, fields, method signatures, and literal values.
2. **Runtime Constant Pool**: When a class is loaded, the JVM constructs this pool in the **Metaspace / Method Area** from the class file's constant pool.
3. **String Constant Pool (SCP)**: A special pool located inside the **Heap Memory** that stores unique String literals to ensure duplicate strings reuse the same memory address.
4. **Primitive Wrapper Constant Cache (Integer Cache)**: Java pre-caches commonly used primitive wrapper objects to prevent unnecessary object allocation:
   - `Byte`, `Short`, `Integer`, `Long`: Cached from **`-128` to `127`**.
   - `Character`: Cached from **`0` to `127`**.
   - `Boolean`: Pre-instantiated `Boolean.TRUE` and `Boolean.FALSE`.


In [ ]:
// Demonstrating Primitive Wrapper Constant Pool Cache vs Heap Allocation

// 1. Within Cache Range (-128 to 127)
Integer numA = 100; // Autoboxed via Integer.valueOf(100) -> pulled from cache
Integer numB = 100;
System.out.println("numA == numB (100 in Cache): " + (numA == numB)); // true (same reference)

// 2. Outside Cache Range (> 127)
Integer numX = 200; // New object allocated on Heap
Integer numY = 200;
System.out.println("numX == numY (200 outside Cache): " + (numX == numY)); // false (different heap references)
System.out.println("numX.equals(numY) (value comparison): " + numX.equals(numY)); // true

// 3. String Constant Pool (SCP) Interning
String s1 = "CDAC";
String s2 = new String("CDAC");
System.out.println("s1 == s2 (Literal vs new Object): " + (s1 == s2)); // false
System.out.println("s1 == s2.intern() (Explicit SCP lookup): " + (s1 == s2.intern())); // true
